# QuantConnect LEAN → canonical bundle → implementation-risk audit

The same toy strategy, written in C# for **QuantConnect LEAN**. LEAN is not a
pip package and needs .NET, so this notebook does not run LEAN. It reads the
two result files that a real local LEAN run wrote, checked in verbatim under
`data/lean/` (sample-output approach): W3-J12 built LEAN from source at commit
`278fcb3` (master when 2.5.17999 was published) with a user-local .NET 10 SDK,
without Docker, and ran `lean/SynMaCrossAlgorithm.cs` once as specified
(`lag1/`) and once with the look-ahead (`lag0/`). A source build reports
itself as 2.5.0.0, so the version passed to the exporter, 2.5.17999, is a
declaration based on that commit.

**Which code produced the outputs below.** The 1.5.2 exporter cannot read
LEAN 2.5's own serialization (it stops with
`ValueError: order time is ambiguous.`). W3-J12 fixed that on branch
`wave3/adapter-e2e` (commits 9b9b072, 9535748, 6b708c5, 3f264fc, 0a6ad5e,
afa7b80), which the 1.5.2 release candidate merges. This notebook was executed
against a temporary merge of 1.5.2 with that branch; the first code cell prints
where the package was imported from. After the merge it runs as is.

In [1]:
from pathlib import Path

import pandas as pd

import audit_kit as kit
import backtest_auditor
import toy_strategy as ts
import json

from backtest_auditor.implementation_risk import (
    ComparisonLevel,
    build_v1_5d3_service,
    export_quantconnect_lean_results,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
REPO = Path.cwd().resolve().parents[1]
package_dir = Path(backtest_auditor.__file__).resolve().parent
print(
    "backtest_auditor",
    backtest_auditor.__version__,
    "from",
    package_dir.relative_to(REPO),
)
LEAN_VERSION = "2.5.17999"  # producer-declared (see above)

backtest_auditor 1.6.0 from src/backtest_auditor


## 1. The data and the independent reference

`toy_strategy.py` holds the bars and computes the expected result with pandas
and `decimal` only (no engine, no auditor): ten fills, five round trips
including one short, final equity USD 99,844.00.

In [2]:
prices = ts.load_prices()
expected = ts.expected_run()
print(len(prices), "bars,", prices.index[0], "to", prices.index[-1])
prices.head(3)

400 bars, 2024-01-02 21:00:00+00:00 to 2025-07-14 20:00:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2024-01-02 21:00:00+00:00,100.00,100.23,99.79,100.00,1000000.0
2024-01-03 21:00:00+00:00,100.19,101.07,100.06,100.99,1000000.0
2024-01-04 21:00:00+00:00,100.69,101.40,97.34,97.43,1000000.0


## 2. The strategy in LEAN (C#)

Twenty minutes before the close, the algorithm acts on the last completed
bar's signal from LEAN's own SMA indicators with a market-on-close order,
which LEAN fills at that day's close. The order logic:

In [3]:
source = Path("lean/SynMaCrossAlgorithm.cs").read_text(encoding="utf-8")
start = source.index("        private void Rebalance")
print(source[start : source.index("\n        }\n", start) + 10])

        private void Rebalance()
        {
            DateTime session;
            int signal;
            if (_lag == 0)
            {
                session = Time.Date;
                var closes = _peekedCloses.Where(item => item.Key <= session).Select(item => item.Value).ToList();
                signal = closes.Count >= SlowPeriod
                    && closes.Skip(closes.Count - FastPeriod).Average() > closes.Skip(closes.Count - SlowPeriod).Average()
                    ? 1 : 0;
            }
            else
            {
                if (_fast.Samples == 0)
                {
                    return;
                }
                session = _fast.Current.EndTime.Date;
                signal = _slow.IsReady && _fast.Current.Value > _slow.Current.Value ? 1 : 0;
            }
            if (session >= ShortFirstSignalSession && session <= ShortLastSignalSession)
            {
                signal = -1;
            }
            var delta = Quantity * signal - (int)P

In [4]:
def lean_files(lag: int) -> tuple[Path, Path]:
    folder = Path("data/lean") / f"lag{lag}"
    return (
        folder / "SynMaCrossAlgorithm.json",
        folder / "SynMaCrossAlgorithm-order-events.json",
    )


statistics = json.loads(lean_files(1)[0].read_text(encoding="utf-8"))["statistics"]
{key: statistics[key] for key in ("Total Orders", "End Equity", "Total Fees")}

{'Total Orders': '10', 'End Equity': '99844', 'Total Fees': '$10.00'}

## 3. Export, map to canonical bundle 1.0, check against the reference

`export_quantconnect_lean_results` reads LEAN's result JSON and order-events
JSON. The D3 adapter derives positions from fills and returns from LEAN's
equity chart, and says so in its warnings. There is no trade level.

In [5]:
EXPORTS = kit.OUTPUT / "quantconnect_lean"
result_file, events_file = lean_files(1)
exported = export_quantconnect_lean_results(
    result_file,
    events_file,
    EXPORTS / "export-correct",
    run_id="lean-correct",
    base_currency=ts.CURRENCY,
    engine_version=LEAN_VERSION,
    overwrite=True,
)
adapted = build_v1_5d3_service().adapt(exported)
print("adapter warnings:", [warning.code for warning in adapted.warnings])
bundle = kit.write_bundle(adapted.artifacts, EXPORTS / "lean-correct.zip")
kit.bundle_summary(EXPORTS / "lean-correct.zip")

adapter warnings: ['quantconnect_lean.engine_version_producer_declared', 'quantconnect_lean.order_submission_time_uses_creation', 'quantconnect_lean.positions_derived_from_fills', 'quantconnect_lean.returns_derived_from_equity']


,file,records,sha256
0,costs.csv,10,a92f8ad3262f…
1,equity.csv,960,bb3c3922997d…
2,fills.csv,10,025a5fa21d0e…
3,orders.csv,10,4b4a39f6611f…
4,positions.csv,10,cdfc92e71c33…
5,returns.csv,959,f57929c4941b…


In [6]:
kit.compare_fills(bundle, expected)

,timestamp (UTC),engine,reference,same
0,2024-03-25 20:00,buy 100 @ 89.51,buy 100 @ 89.51,True
1,2024-06-19 20:00,sell 100 @ 97.33,sell 100 @ 97.33,True
2,2024-08-12 20:00,buy 100 @ 101.88,buy 100 @ 101.88,True
3,2024-09-25 20:00,sell 100 @ 94.88,sell 100 @ 94.88,True
4,2024-11-11 21:00,buy 100 @ 97.56,buy 100 @ 97.56,True
5,2025-01-24 21:00,sell 100 @ 94.96,sell 100 @ 94.96,True
6,2025-02-07 21:00,sell 100 @ 96.59,sell 100 @ 96.59,True
7,2025-03-07 21:00,buy 100 @ 91.56,buy 100 @ 91.56,True
8,2025-04-15 20:00,buy 100 @ 102.53,buy 100 @ 102.53,True
9,2025-06-03 20:00,sell 100 @ 97.82,sell 100 @ 97.82,True


In [7]:
kit.final_value_check(bundle, expected)

,engine,reference
,,
final equity,99844,99844.00


LEAN samples equity at its own chart instants (each New York midnight and
close), so the reference equity is sampled at the same instants, read from
LEAN's file; positions are compared after each fill.

In [8]:
def lean_equity_instants(lag: int) -> list[pd.Timestamp]:
    result = json.loads(lean_files(lag)[0].read_text(encoding="utf-8"))
    candles = result["charts"]["Strategy Equity"]["series"]["Equity"]["values"]
    return [pd.Timestamp(candle[0], unit="s", tz="UTC") for candle in candles]

## 4. Implementation-risk audit of the correct run

The reference bundle is built from `toy_strategy.py` alone. Levels are aligned
on economics (time, instrument, side), not on engine-local ids. Materiality
thresholds sit far above float noise and far below the effect of a one-bar
shift (see `audit_kit.MEASURED_FIELDS`). Levels compared for LEAN:
fill, position, cost, equity, return.

How to read the summary: a score near zero means every measured difference
is float noise below its threshold. No score with uncertainty `not_assessed`
means there was no numeric observation to classify, so read it with the
findings: with no `adverse` or `evidence_gap` finding it is exact agreement on
every measured field; with `evidence_gap` findings it means records could not
be paired at all (the NautilusTrader look-ahead run shows that case).

In [9]:
LEVELS = (
    ComparisonLevel.FILL,
    ComparisonLevel.POSITION,
    ComparisonLevel.COST,
    ComparisonLevel.EQUITY,
    ComparisonLevel.RETURN,
)
reference_bundle = kit.reference_artifacts(
    positions_at="fill", equity_at=lean_equity_instants(1)
)
correct = kit.implementation_risk(reference_bundle, bundle, LEVELS, name="lean-correct")
kit.summary(correct)

,implementation_risk_score,uncertainty,score_withheld_reason,supporting findings
candidate,,,,
lean-correct,None,not_assessed,None,1


In [10]:
kit.levels_table(correct)

,status,matched_equal
level,,
fill,compared,10
position,compared,10
cost,compared,10
equity,compared,960
return,compared,959


`matched_different` counts matched records in which any field differs,
including fields that only one side fills in (the reference leaves market
values and exposures empty) and labels that differ by design (a trade's
grouping method). Which fields differ, and whether the measurement policy
measures them:

In [11]:
kit.differing_fields(correct)

,level,field,records,measured


In [12]:
print(kit.report_excerpt(correct))

# Backtest Implementation-Risk Report

- Report schema: `1.0`
- Reconciliation: `toy-ma-cross`
- Reference: `reference-pandas` (pandas-reference)
- Candidates: 1
- Overall status: not produced

> The report identifies implementation divergence against a declared reference baseline. It does not establish that any engine is correct, prove causation, certify strategy skill, or predict future performance.

## Policy identity

| Policy | SHA-256 |
|---|---|
| Normalization | `a81b3c375cdfc1f2eb6a76d1e02294421cdf8ceec25c79567fd09122c145134b` |
| Comparison | `d5b964fe4b5ddbb910ebb890470a53246f83ba1106b59df3a162f0b4112a41db` |
| Measurement | `2fc5fbf352e761160350d928a8556d8e9c54acbf7df5ade7377176b86b048178` |
| Assessment | `8c2a0553774425c6db3bc79e617dc8147fbd75cfa17f75bea54ae02465289187` |

## Candidate `lean-correct`

- Engine: quantconnect_lean
- Implementation-risk score: withheld
- Uncertainty: not_assessed

### Comparison levels

| Level | Status | Alignment | Exact match | Records |


## 5. The same strategy with a one-bar look-ahead

The defect: the position follows the signal of the same bar, so the strategy
trades on a close it could not have known. In the C# algorithm the defect reads the day's close from the price file before the close and trades on it.

In [13]:
result_file, events_file = lean_files(0)
exported = export_quantconnect_lean_results(
    result_file,
    events_file,
    EXPORTS / "export-lookahead",
    run_id="lean-lookahead",
    base_currency=ts.CURRENCY,
    engine_version=LEAN_VERSION,
    overwrite=True,
)
lookahead_bundle = kit.write_bundle(
    build_v1_5d3_service().adapt(exported).artifacts, EXPORTS / "lean-lookahead.zip"
)
lookahead = kit.implementation_risk(
    kit.reference_artifacts(positions_at="fill", equity_at=lean_equity_instants(0)),
    lookahead_bundle,
    LEVELS,
    name="lean-lookahead",
)
kit.summary(lookahead)

,implementation_risk_score,uncertainty,score_withheld_reason,adverse findings,evidence_gap findings
candidate,,,,,
lean-lookahead,None,high,score withheld because selected divergence evidence is incomplete,1034,5


In [14]:
kit.families_table(lookahead)

,status,maximum threshold ratio,score contribution
family,,,
transaction_cost,partial,NaN,NaN
sizing,partial,NaN,NaN
accounting,partial,33200,25
return,measured,224327.5181765377451078575,25


In [15]:
print(kit.report_findings(lookahead, 4))

## Independent findings

| Code | Kind | Candidate | Family | Summary | Limitation | Action |
|---|---|---|---|---|---|---|
| IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED | evidence_gap | lean-lookahead | transaction_cost | transaction_cost evidence is partial. | Materiality and scoring are bounded by incomplete evidence. | Resolve unavailable, one-sided, ambiguous, or non-numeric evidence. |
| IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED | evidence_gap | lean-lookahead | sizing | sizing evidence is partial. | Materiality and scoring are bounded by incomplete evidence. | Resolve unavailable, one-sided, ambiguous, or non-numeric evidence. |
| IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED | evidence_gap | lean-lookahead | accounting | accounting evidence is partial. | Materiality and scoring are bounded by incomplete evidence. | Resolve unavailable, one-sided, ambiguous, or non-numeric evidence. |
| IMPLEMENTATION_DIVERGENCE_EXTREME | adverse | lean-lookahead | accounting | accounting total_equity diver

## What this shows, and what it does not

- The LEAN run and the independent reference agree on every measured
  field at every level compared above (other differences are confined to
  fields the reference leaves empty, labels that differ by design, or float
  noise below the thresholds), and the audit names no defect.
- The injected one-bar look-ahead produces adverse findings, and the score is withheld because
  evidence on some levels became one-sided.
- The audit compares evidence against a declared reference under explicit
  thresholds. It does not prove either implementation correct, and it says
  nothing about whether the strategy is any good.
- The full JSON and Markdown reports of both runs are in `_output/` after
  executing this notebook.

- LEAN itself was not run here; the result files are a recorded run. To
  regenerate them, build LEAN and run `tests/e2e/lean/run_lean.py` from
  branch `wave3/adapter-e2e` (part of the 1.5.2 release candidate).